In [19]:
from ultralytics import YOLO
import os
import glob
import shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── PATHS ──────────────────────────────────────────────────────
BASE_PATH    = r"C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase"
DATASET_PATH = os.path.join(BASE_PATH, "wasp_dataset", "split")
DATA_YAML    = os.path.join(BASE_PATH, "data.yaml")
RUNS_PATH    = os.path.join(BASE_PATH, "runs", "detect")
# ───────────────────────────────────────────────────────────────

print("✅ Paths set")
print(f"   Base     : {BASE_PATH}")
print(f"   Dataset  : {DATASET_PATH}")
print(f"   YAML     : {DATA_YAML}")
print(f"   Runs     : {RUNS_PATH}")

✅ Paths set
   Base     : C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase
   Dataset  : C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split
   YAML     : C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\data.yaml
   Runs     : C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\runs\detect


In [20]:
train_count = len(glob.glob(os.path.join(DATASET_PATH, "train", "images", "*")))
val_count   = len(glob.glob(os.path.join(DATASET_PATH, "val",   "images", "*")))
test_count  = len(glob.glob(os.path.join(DATASET_PATH, "test",  "images", "*")))
yaml_exists = os.path.exists(DATA_YAML)

print("📁 Dataset Check:")
print(f"   Train  : {train_count} images")
print(f"   Val    : {val_count} images")
print(f"   Test   : {test_count} images")
print(f"   YAML   : {'✅ Found' if yaml_exists else '❌ Not found — check path'}")

if not yaml_exists:
    print(f"\n⚠️ data.yaml not found at:\n   {DATA_YAML}")

📁 Dataset Check:
   Train  : 4000 images
   Val    : 500 images
   Test   : 500 images
   YAML   : ✅ Found


In [21]:
# ⚠️ WARNING — this deletes ALL previous runs
# Only run this cell when you want to start completely fresh
# Comment out if you want to keep old results

if os.path.exists(RUNS_PATH):
    shutil.rmtree(RUNS_PATH)
    print("✅ Old runs deleted — ready for fresh training")
else:
    print("No old runs found — already clean")

✅ Old runs deleted — ready for fresh training


Train YOLOv8n

In [22]:
model = YOLO("yolov8n.pt")

model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=416,
    batch=4,
    name="wasp_ppe_v1",
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    project=RUNS_PATH
)

print("\n✅ Training complete")
print(f"   Results saved to: {os.path.join(RUNS_PATH, 'wasp_ppe_v1')}")

New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.235  Python-3.12.12 torch-2.9.1+cu126 CPU (AMD Ryzen 5 5625U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_sc

In [23]:
weights = glob.glob(os.path.join(RUNS_PATH, "*", "weights", "best.pt"))

if not weights:
    print("❌ No best.pt found")
    print("   Make sure training completed successfully")
else:
    print("✅ Models found:")
    for w in weights:
        size_mb = os.path.getsize(w) / (1024 * 1024)
        print(f"   {w}  ({size_mb:.2f} MB)")
    
    # Use the most recent one
    best_model_path = sorted(weights)[-1]
    print(f"\n🎯 Using: {best_model_path}")

✅ Models found:
   C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\runs\detect\wasp_ppe_v1\weights\best.pt  (5.93 MB)

🎯 Using: C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\runs\detect\wasp_ppe_v1\weights\best.pt


Validate trained model

In [24]:
model = YOLO(best_model_path)

metrics = model.val(data=DATA_YAML)

print("\n📊 Validation Results:")
print(f"   mAP50       : {metrics.box.map50:.4f}")
print(f"   mAP50-95    : {metrics.box.map:.4f}")
print(f"   Precision   : {metrics.box.mp:.4f}")
print(f"   Recall      : {metrics.box.mr:.4f}")
print(f"\n   Meaning:")
print(f"   mAP50 above 0.75 = good for MVP ✅")

Ultralytics 8.3.235  Python-3.12.12 torch-2.9.1+cu126 CPU (AMD Ryzen 5 5625U with Radeon Graphics)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1257.7331.7 MB/s, size: 263.4 KB)
val: Scanning C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\val\labels.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 495.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.4it/s 22.6s0.7ss
                   all        500       2593      0.954      0.585      0.638      0.414
                helmet        448       1902      0.936       0.88      0.944      0.608
                  head        103        597      0.926      0.876      0.937      0.614
                person         14         94          1          0     0.0324     0.0204
Speed: 0.5ms preprocess, 35.2ms inference, 0.0ms loss, 0.

Predict test images

In [25]:
model.predict(
    source=os.path.join(DATASET_PATH, "test", "images"),
    conf=0.5,
    save=True,
    name="wasp_predictions",
    exist_ok=True,
    project=RUNS_PATH
)

pred_path = os.path.join(RUNS_PATH, "wasp_predictions")
result_images = (
    glob.glob(os.path.join(pred_path, "*.jpg")) +
    glob.glob(os.path.join(pred_path, "*.png"))
)

print(f"✅ Predictions saved to: {pred_path}")
print(f"   Total predicted images: {len(result_images)}")


image 1/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\test\images\hard_hat_workers100.png: 416x416 5 helmets, 52.1ms
image 2/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\test\images\hard_hat_workers1000.png: 416x416 2 helmets, 41.9ms
image 3/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\test\images\hard_hat_workers1009.png: 416x416 3 helmets, 41.7ms
image 4/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\test\images\hard_hat_workers1012.png: 416x416 4 helmets, 39.5ms
image 5/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\test\images\hard_hat_workers1020.png: 416x416 7 helmets, 41.6ms
image 6/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcase\wasp_dataset\split\test\images\hard_hat_workers1023.png: 416x416 6 helmets, 41.7ms
image 7/500 C:\Users\User\OneDrive\STUDY\UTM KL\UTM\Project\AI_Showcas

Display prediction results

In [26]:
pred_path = os.path.join(RUNS_PATH, "wasp_predictions")

result_images = (
    glob.glob(os.path.join(pred_path, "*.jpg")) +
    glob.glob(os.path.join(pred_path, "*.png"))
)

if not result_images:
    print("❌ No prediction images found")
    print(f"   Check folder: {pred_path}")
else:
    print(f"✅ Showing 6 sample predictions")

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for i in range(6):
        if i < len(result_images):
            img = mpimg.imread(result_images[i])
            axes[i].imshow(img)
            axes[i].axis("off")
            axes[i].set_title(f"Sample {i+1}", fontsize=12)
        else:
            axes[i].axis("off")

    plt.suptitle("WASP — PPE Detection Results", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

✅ Showing 6 sample predictions


<Figure size 1500x1000 with 6 Axes>

Test webcam later

In [29]:
model = YOLO(best_model_path)

print("🎥 Starting live webcam detection")
print("   Press Q to stop\n")
print("   Detection classes:")
print("   🟢 helmet = worker wearing hard hat (SAFE)")
print("   🔴 head   = worker WITHOUT hard hat (VIOLATION)")
print("   🔵 person = person detected in zone\n")

model.predict(
    source=0,
    conf=0.5,
    show=True,
    stream=True
)

🎥 Starting live webcam detection
   Press Q to stop

   Detection classes:
   🟢 helmet = worker wearing hard hat (SAFE)
   🔴 head   = worker WITHOUT hard hat (VIOLATION)
   🔵 person = person detected in zone



<generator object BasePredictor.stream_inference at 0x000001CE129BFD00>